# Week 3 V1 Data Pipeline: Wattlah

*SAIA3113 AI Engineering, Topic 3 Lab. Muhammad Uwais Bin Zulkarnain (A24AI0114).*

This notebook builds the first data pipeline for Wattlah. It feeds the two data components of the Week 2 V1 architecture:

| Week 2 component | Pathway | Source files | Prepared output |
|---|---|---|---|
| Knowledge Source | A, documents (PDF) | S02 to S06: ST and TNB statements | `data/processed/prepared_text.jsonl` |
| Tariff Rate Table | B, tables (CSV) | S01 tariff schedule, transcribed to CSV and verified | `data/processed/prepared_tariff_rates.csv`, `prepared_relief_rules.csv` |

Requirements driving this week: **FR1, NFR1** (rates must be exact), **FR4, RAI1** (every item needs its effective period), **FR5** (source and page for every chunk), **NFR5** (rates kept as data), **RAI4** (no personal data in prepared outputs).

**Run order:** Cell 1 → upload files → Cell 3 → Cell 4 → Cell 4A → Part A inspection → Cells 5A1 to 5A2 → B3 (A) → Cell 5B → B3 (B) → B4.1 → 6A → 6B → 7 → 8A → 8B → 9.

**Colab upload:** put the 6 PDFs plus `source_registry.csv`, `tariff_domestic_rp4.csv` and `relief_rules.csv` in `week3_project/data/raw`. The PDFs come from the URLs in `source_registry.csv`; Cell 3 checks each file's SHA-256 against the registry.

## Part A: Start with the Data

In [1]:
# Cell 1: workspace setup (run first, and again after a runtime restart)
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    BASE_DIR = Path("/content/week3_project")
elif os.environ.get("WEEK3_BASE"):
    BASE_DIR = Path(os.environ["WEEK3_BASE"])
else:
    BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW, PROCESSED, BASE, OUT = RAW_DIR, PROCESSED_DIR, BASE_DIR, PROCESSED_DIR
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


Cell 2 (dummy ZIP extraction) is not used: this project uses its own sources.

In [2]:
# Cell 3: verify inputs against the source registry
import csv, hashlib

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

files = sorted(f for f in RAW_DIR.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f" {f.relative_to(RAW_DIR)}  {f.stat().st_size:>9,} bytes")
assert files, "Upload the source files into data/raw first."

with open(RAW_DIR / "source_registry.csv", encoding="utf-8") as fh:
    REGISTRY = {row["source_id"]: row for row in csv.DictReader(fh)}

print("\nSource identity check")
missing, mismatched = [], []
for sid, row in REGISTRY.items():
    path = RAW_DIR / row["filename"]
    if not path.is_file():
        missing.append(sid)
        status = "MISSING"
    elif sha256(path) != row["sha256"]:
        mismatched.append(sid)
        status = "CHECKSUM MISMATCH"
    else:
        status = "ok"
    print(f" {sid}  {row['filename']:<40} {status}")

registered = {r["filename"] for r in REGISTRY.values()} | {"source_registry.csv", "tariff_domestic_rp4.csv", "relief_rules.csv"}
unregistered = [f.name for f in files if f.name not in registered]
assert not missing, f"Missing sources: {missing}"
assert not mismatched, f"File differs from the registered version: {mismatched}. Re-download or update the registry."
print("Unregistered files ignored:", unregistered or "none")
print("All registered sources present and unchanged.")

Input count: 9
 relief_rules.csv      1,176 bytes
 source_registry.csv      3,124 bytes
 st_kenyataan_media_2025-06-20.pdf    752,598 bytes
 tariff_domestic_rp4.csv        685 bytes
 tnb_20260915_23_bi.pdf    377,233 bytes
 tnb_20260915_23_bm.pdf    381,930 bytes
 tnb_20260918_24_bm.pdf  2,790,930 bytes
 tnb_20261002_25_bi.pdf    680,523 bytes
 tnb_jadual_tarif_2025-07-02.pdf  2,964,276 bytes

Source identity check
 S01  tnb_jadual_tarif_2025-07-02.pdf          ok
 S02  st_kenyataan_media_2025-06-20.pdf        ok
 S03  tnb_20260915_23_bi.pdf                   ok
 S04  tnb_20260915_23_bm.pdf                   ok
 S05  tnb_20260918_24_bm.pdf                   ok
 S06  tnb_20261002_25_bi.pdf                   ok
Unregistered files ignored: none
All registered sources present and unchanged.


In [3]:
# Cell 4: source selection and lineage utilities (lab helpers)
import json
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(p for p in RAW_DIR.rglob("*") if p.is_file()
                     and (filename is None or p.name == filename)
                     and (suffixes is None or p.suffix.lower() in suffixes))
    if len(matches) != 1:
        raise ValueError(f"Select exactly one source, found {[p.name for p in matches]}")
    return matches[0]

def source_hash(path):
    return sha256(path)

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records,
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target.relative_to(BASE_DIR))
    return manifest

def source_path(sid):
    return select_source(REGISTRY[sid]["filename"])

print("Cell 4 ready: select_source(), source_hash(), save_manifest(), source_path()")

Cell 4 ready: select_source(), source_hash(), save_manifest(), source_path()


In [4]:
# Cell 4A: common lineage registry (lab code)
lineage_entries = {}

def register_lineage(source, output, processing):
    source, output = Path(source).resolve(), Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip() or processing.strip().upper().startswith("REPLACE"):
        raise ValueError("A real processing description is required.")
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    lineage_entries[(str(source), str(output))] = {"source": source, "output": output, "processing": processing.strip()}
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready. Registered entries:", len(lineage_entries))

Common lineage registry ready. Registered entries: 0


### A3: Inspect what was actually loaded

Page-level evidence for every source: page count, characters extracted per page, and the first characters of text. A page that loads but returns little or no text is a **silent failure**, so it is flagged here instead of being trusted.

In [5]:
# A3: page-level inspection of all six PDFs
!pip -q install --root-user-action=ignore pypdf
import re, logging
from pypdf import PdfReader
logging.getLogger("pypdf").setLevel(logging.ERROR)

def page_texts(path, mode="layout"):
    reader = PdfReader(str(path))
    out = []
    for page in reader.pages:
        raw = page.extract_text(extraction_mode=mode) or ""
        out.append([re.sub(r"[ \t]+", " ", l).strip() for l in raw.splitlines() if l.strip()])
    return out

INSPECTION = {}
print(f"{'ID':<4}{'lang':<5}{'pages':>5}  chars per page")
for sid, row in REGISTRY.items():
    pages = page_texts(source_path(sid))
    chars = [sum(len(l) for l in p) for p in pages]
    INSPECTION[sid] = {"pages": len(pages), "chars": chars}
    assert len(pages) == int(row["pages"]), f"{sid}: expected {row['pages']} pages, loaded {len(pages)}"
    print(f"{sid:<4}{row['language']:<5}{len(pages):>5}  {chars}")

ID  lang pages  chars per page


S01 ms       7  [1394, 1512, 1354, 1425, 676, 1504, 1401]


S02 ms       5  [1758, 1976, 1977, 1687, 484]
S03 en       3  [2216, 2443, 827]


S04 ms       3  [2257, 2256, 1066]


S05 ms       4  [2104, 2028, 939, 1044]
S06 en       3  [1854, 1600, 34]


In [6]:
# A3: representative samples, including the problem pages
def show(sid, page, first=None, last=None, mode="layout"):
    lines = page_texts(source_path(sid), mode)[page - 1]
    print(f"--- {sid} page {page} ({REGISTRY[sid]['filename']}) ---")
    for l in lines[first:last]:
        print("  |", l)
    print()

show("S03", 1, 0, 12)              # clean English press release
show("S01", 1, 0, 9, mode="plain")  # tariff schedule, Domestik Am block
show("S06", 1, 18, 26)              # eligibility table flattened into lines
show("S06", 3)                      # page with no content
show("S02", 5)                      # appendix: only a caption and the footer survive
show("S05", 4, 5, 14)               # infographic: broken reading order

--- S03 page 1 (tnb_20260915_23_bi.pdf) ---
  | PRESS RELEASE
  | S.A. 2026/09/15_23 (HQ)
  | ADDRESSING ELECTRICITY BILL CONCERNS: AFA MECHANISM ENSURES TRANSPARENCY
  | AND FAIR COST-SHARING SINCE JULY 2025
  | Tenaga Nasional Berhad (TNB) notes concern raised by certain segments of customers
  | regarding electricity bills, particularly on the Automatic Fuel Adjustment (AFA) component.
  | TNB understands the pressures of the cost of living and would like to provide an explanation
  | on how the AFA mechanism works, as well as the measures implemented to protect
  | customers.
  | AFA was introduced on 1 July 2025 following the restructuring of the electricity tariff,
  | replacing the Imbalance Cost Pass-Through (ICPT) mechanism. As such, AFA is not a new
  | charge and does not generate profits for TNB. AFA functions as a two-way mechanism to



--- S01 page 1 (tnb_jadual_tarif_2025-07-02.pdf) ---
  | Jadual Tarif Elektrik di Semenanjung
  | Bagi Tempoh 1 Julai 2025 Hingga 31 Disember 2027
  | Kadar tarif untuk Kategori Umum ditetapkan seperti berikut:
  | Domestik Am
  | Kategori Tarif Unit Kadar Baharu
  | Caj Tenaga - Untuk penggunaan ≤1500 kW sen/kWj 27.03
  | Caj Tenaga - Untuk penggunaan >1500 kWj sen/kWj 37.03
  | Caj Kapasiti - Untuk semua kWj sen/kWj 4.55
  | Caj Rangkaian - Untuk semua kWj sen/kWj 12.85

--- S06 page 1 (tnb_20261002_25_bi.pdf) ---
  | Bil adjustment eligibility schedule for October 2026 bill
  | Total Consumption (kWh) in Bill Adjustment Eligibility
  | September 2026
  | 600kWh and below No adjustment will be applied to the October 2026 bill
  | 601kWh to 800kWh Eligible for adjustments to the AFA, Retail Charge and
  | 8% Service Tax
  | 801kWh and above Eligible for adjustment to the 8% Service Tax
  | Note: The above schedule applies to September 2026 bills based on actual meter readings.

--- S0

--- S02 page 5 (st_kenyataan_media_2025-06-20.pdf) ---
  | LAMPIRAN
  | Contoh perbandingan bil selepas penstrukturan
  | semula tarif
  | MENGENAI SURUHANJAYA TENAGA
  | Suruhanjaya Tenaga (ST) merupakan badan kawal selia yang bertanggungjawab untuk mengawal selia sektor
  | tenaga, khususnya industri pembekalan tenaga elektrik dan gas berpaip di Semenanjung Malaysia dan Wilayah
  | Persekutuan Labuan . Suruhanjaya Tenaga memastikan bahawa pembekalan elektrik dan gas berpaip kepada
  | pengguna adalah terjamin, berdaya harap, selamat dan pada harga yang berpatutan.
  | 5 of 5



--- S05 page 4 (tnb_20260918_24_bm.pdf) ---
  | ARAS PERLINDUNGAN DIPERTINGKAT
  | 60 0 kWj 80 0kWj
  | sebulan sebulan
  | Berkuat kuasa mulai penggunaan September hingga Disember 2026
  | LEBIH SEHINGGA HAMPIR
  | 8 JUTA 1 JUTA90%
  | pengguna domestik pengguna domestik pengguna domestik tambahan
  | dilindungi menerima manfaat dilindungi berbanding ambang
  | sebelumnya



**What the inspection shows** (recorded in `docs/week3_data_inventory.md`):

- All 6 files load, page counts match the registry, and every file matches its registered SHA-256.
- **S02 page 5** returns only a caption and footer. The page is an image of a worked bill example; its numbers are not extractable without OCR. This is a silent failure: the page loads but carries no usable text.
- **S06 page 3** has only the reference number and page number: no content.
- **S06 page 1** contains an eligibility table that extracts as lines. Rows stay in order, but the column structure is lost.
- **S05 page 4** is an infographic that repeats pages 1 to 2. Its reading order is broken (for example `8 JUTA 1 JUTA90%`).
- **S01 page 1** uses plain extraction for the table: the Domestik Am rows come out with label, unit and value on one line. The printed label has a typo in the source itself: `≤1500 kW` where every other row says `kWj`.
- The re-saved S02 file raises PDF cross-reference warnings in pypdf but still loads every page.

## Part B: Prepare the Data

### B1: Selected pathways

- **Pathway A (PDF)** for the five statements S02 to S06. They hold the rules and explanations the Knowledge Source needs (FR2, FR4, FR5).
- **Pathway B (CSV)** for the Tariff Rate Table. S01 is a table, and the calculator needs exact numbers (FR1, NFR1), so its rows are transcribed into `tariff_domestic_rp4.csv`. Thresholds and the monthly AFA rate from S01, S03 and S06 go into `relief_rules.csv`. Every row carries its source and page, and Cell 5B checks each value against the PDF text.
- Only the Domestik Am tariff is transcribed. ToU, non domestic, solar and Sabah or Sarawak tariffs are out of V1 scope (Week 2, Section 0).

In [7]:
# Cell 5A1 (part 1): extraction and justified cleaning for Pathway A
DOC_SOURCES = [sid for sid, r in REGISTRY.items() if r["pipeline"] == "documents"]

FURNITURE = [re.compile(p) for p in (
    r"^Page \d+ of \d+$", r"^\d+ of \d+$",
    r"^S\.A\. \d{4}/\d{2}/\d{2}_\d+ \(HQ\)$",
    r"^(PRESS RELEASE|SIARAN AKHBAR|KENYATAAN MEDIA)$")]
ST_LETTERHEAD = [re.compile(p) for p in (
    r"^www\.st\.gov\.my$", r"^Suruhanjaya Tenaga$", r"^12, Jalan Tun Hussein",
    r"^62100 Putrajaya", r"^T : \+603", r"^Toll Free", r"^\d{1,2} JUN 2025$")]
ST_FOOTER = re.compile(r"^MENGENAI SURUHANJAYA TENAGA$")
END_MARKER = re.compile(r"^(\*{5}|\*{3}TAMAT\*{3})$")
MIN_PAGE_CHARS = 80

def clean_pages(pages):
    log = {"furniture": 0, "st_letterhead": 0, "st_footer": 0, "after_end_marker": 0}
    cleaned, ended = [], False
    for page_no, lines in enumerate(pages, 1):
        kept, in_footer = [], False
        for line in lines:
            if ended:
                log["after_end_marker"] += 1
            elif END_MARKER.match(line):
                ended = True
                log["after_end_marker"] += 1
            elif in_footer or ST_FOOTER.match(line):
                in_footer = True
                log["st_footer"] += 1
            elif any(p.match(line) for p in FURNITURE):
                log["furniture"] += 1
            elif any(p.match(line) for p in ST_LETTERHEAD):
                log["st_letterhead"] += 1
            else:
                kept.append(line)
        cleaned.append((page_no, kept))
    return cleaned, log

CLEANED, CLEAN_LOG, PAGE_STATUS = {}, {}, {}
print(f"{'ID':<4}{'lines removed by rule':<72}content pages")
for sid in DOC_SOURCES:
    CLEANED[sid], CLEAN_LOG[sid] = clean_pages(page_texts(source_path(sid)))
    for page_no, lines in CLEANED[sid]:
        size = sum(len(l) for l in lines)
        PAGE_STATUS[(sid, page_no)] = "content" if size >= MIN_PAGE_CHARS else "no usable text"
    content = [p for (s, p), v in PAGE_STATUS.items() if s == sid and v == "content"]
    print(f"{sid:<4}{str(CLEAN_LOG[sid]):<72}{content}")

ID  lines removed by rule                                                   content pages


S02 {'furniture': 1, 'st_letterhead': 7, 'st_footer': 18, 'after_end_marker': 17}[1, 2, 3, 4]


S03 {'furniture': 4, 'st_letterhead': 0, 'st_footer': 0, 'after_end_marker': 18}[1, 2]


S04 {'furniture': 4, 'st_letterhead': 0, 'st_footer': 0, 'after_end_marker': 18}[1, 2]


S05 {'furniture': 4, 'st_letterhead': 0, 'st_footer': 0, 'after_end_marker': 43}[1, 2]
S06 {'furniture': 4, 'st_letterhead': 0, 'st_footer': 0, 'after_end_marker': 16}[1, 2]


**Cleaning rules and why each one is justified**

| Rule | Removes | Why |
|---|---|---|
| Whitespace | Runs of spaces from justified text (layout extraction) | Layout mode keeps words intact (plain mode broke `customers` into `c ustomers`) but pads justified lines with spaces |
| Page furniture | `Page x of y`, reference number lines, `PRESS RELEASE` / `SIARAN AKHBAR` / `KENYATAAN MEDIA` | Repeated on every page; the reference number and document type are kept as metadata instead |
| ST letterhead | Address, phone and URL block on S02 page 1 | Contact details of the regulator, not tariff content |
| ST footer | "MENGENAI SURUHANJAYA TENAGA" paragraph on every S02 page | The same boilerplate paragraph 5 times would pollute retrieval |
| After end marker | Everything after `*****` (TNB) or `***TAMAT***` (ST) | Company profile, release line and **press officers' names and phone numbers** (RAI4). Also drops the S05 infographic and S02 appendix, which have no usable text anyway |

Not removed: the TNB CareLine contact block. FR7 needs it to refer users to official TNB channels.

In [8]:
# Cell 5A1 (part 2): rebuild headings and paragraphs, then three chunking strategies
PARA_END = re.compile(r"[.!?:;][\"”’)]*$")
LIST_START = re.compile(r"^(\d+\.\s|\([ivx]+\)\s|[-•]\s)")
SENTENCE_END = re.compile(r"(?<=[.!?])\s+(?=[A-Z(“\"])")

def is_heading(line, next_line=None):
    if len(line) > 80 or PARA_END.search(line) or LIST_START.match(line) or line.endswith(",") or not line[0].isupper():
        return False
    words = [w for w in re.findall(r"[A-Za-z][A-Za-z'’-]*", line) if len(w) > 3]
    if words and sum(w[0].isupper() for w in words) / len(words) >= 0.6:
        return True
    return len(line) <= 65 and bool(next_line) and len(next_line) >= 60 and next_line[0].isupper()

def to_blocks(lines):
    blocks, current = [], []
    prev_closed, last_heading = True, False
    for i, line in enumerate(lines):
        next_line = lines[i + 1] if i + 1 < len(lines) else None
        if current and (prev_closed or LIST_START.match(line)):
            blocks.append(("para", " ".join(current)))
            current = []
        if last_heading and (line.isupper() or len(line) <= 40) and is_heading(line, next_line):
            blocks[-1] = ("heading", blocks[-1][1] + " " + line)
            continue
        if not current and prev_closed and not last_heading and is_heading(line, next_line):
            blocks.append(("heading", line))
            last_heading = True
            continue
        last_heading = False
        current.append(line)
        prev_closed = bool(PARA_END.search(line))
    if current:
        blocks.append(("para", " ".join(current)))
    return blocks

def split_long(text, limit):
    parts, current = [], ""
    for sentence in SENTENCE_END.split(text):
        if current and len(current) + 1 + len(sentence) > limit:
            parts.append(current)
            current = sentence
        else:
            current = f"{current} {sentence}".strip()
    if current:
        parts.append(current)
    return [p[i:i + limit] for p in parts for i in range(0, len(p), limit)]

def structure_chunks(sid, size, overlap_paras=0):
    chunks, heading = [], None
    for page_no, lines in CLEANED[sid]:
        if PAGE_STATUS.get((sid, page_no)) != "content":
            continue
        body = []
        def flush(keep=0):
            if body:
                chunks.append({"page": page_no, "section": heading, "body": list(body)})
                carry = body[-keep:] if keep else []
                body.clear()
                body.extend(carry)
        for kind, text in to_blocks(lines):
            if kind == "heading":
                flush()
                heading = text
                continue
            room = size - (len(heading) + 1 if heading else 0)
            for piece in (split_long(text, room) if len(text) > room else [text]):
                if body and len("\n".join(body)) + 1 + len(piece) > room:
                    flush(overlap_paras)
                    while body and len("\n".join(body)) + 1 + len(piece) > room:
                        body.pop(0)
                body.append(piece)
        flush()
    for c in chunks:
        text = "\n".join(c.pop("body"))
        c["text"] = f"{c['section']}\n{text}" if c["section"] else text
    return chunks

def fixed_chunks(s, size=500, overlap=0):
    step = size - overlap
    return [s[i:i + size] for i in range(0, max(len(s) - overlap, 1), step)]

def lab_paragraph_chunks(s, size=500):
    parts, current = [], ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

def chunk_source(sid, size, overlap_paras=0):
    return structure_chunks(sid, size, overlap_paras)

for kind, text in to_blocks(CLEANED["S06"][0][1])[:9]:
    print(f"{kind.upper():<8}{text[:110]}")

HEADING INCREASE IN PROTECTION THRESHOLD TO 800KWH: CUSTOMERS TO RECEIVE PRO-RATED CREDIT IN OCTOBER 2026 BILL
PARA    Following the government’s announcement on 17 September 2026 to increase the protection threshold for Domestic
PARA    With the increase in the protection threshold, Domestic customers with monthly consumption of 800 kWh and belo
PARA    • Automatic Fuel Adjustment (AFA) charge,
PARA    • RM10 Retail Charge, and
PARA    • 8% Service Tax.
PARA    For eligible customers, the adjustment of the above charges for September will be calculated on a pro-rated ba
PARA    Bil adjustment eligibility schedule for October 2026 bill Total Consumption (kWh) in Bill Adjustment Eligibili
HEADING Maintain Energy Efficiency Practices


### B3 (Pathway A): compare chunking strategies on the same cleaned text

Columns: chunk count, mean and max length in characters, chunks under 150 characters, chunks that start mid-word or mid-sentence, rules kept whole, and **extra text** compared with the source (overlap repeats text; structure-aware chunks repeat their section heading).

The test that matters for Wattlah is whether a **rule stays in one chunk with its qualifier**: the threshold with what it exempts, the exemption with its effective period. A chunk that holds "800 kWh and below are exempted" without "September to 31 December 2026" would let the model apply the rule to the wrong month (RAI1). Ten such rules were picked from the five documents, in both languages.

In [9]:
# B3 (Pathway A): comparison
import statistics as st

KEY_FACTS = [
    ("S02", ["ditetapkan pada kadar 45.40 sen/kWj"]),
    ("S02", ["pengguna domestik dengan penggunaan elektrik 1,000 kWj dan ke bawah dijangka tidak terkesan"]),
    ("S02", ["keseluruhan Sabtu, Ahad dan 10 malam sehingga 2 petang pada hari Isnin hingga Jumaat"]),
    ("S03", ["The AFA rate for September 2026 is +3.67 sen/kWh."]),
    ("S03", ["consumption of 600kWh and below are fully exempted from AFA charges",
             "also exempted from the 8% service tax and Retail Charge"]),
    ("S04", ["Kadar AFA bagi September 2026 ialah +3.67 sen/kWj."]),
    ("S04", ["600kWj ke bawah sebulan adalah dikecualikan sepenuhnya daripada caj AFA",
             "cukai perkhidmatan 8% dan Caj Peruncitan"]),
    ("S05", ["penggunaan 800kWj dan ke bawah sebulan akan dikecualikan",
             "bermula penggunaan elektrik September 2026 hingga 31 Disember 2026"]),
    ("S06", ["consumption of 800 kWh and below from September to 31 December 2026 will enjoy full exemption from:",
             "RM10 Retail Charge", "8% Service Tax."]),
    ("S06", ["601kWh to 800kWh Eligible for adjustments to the AFA, Retail Charge and 8% Service Tax",
             "801kWh and above Eligible for adjustment to the 8% Service Tax"]),
]
norm = lambda t: re.sub(r"\s+", " ", t)

def whole_text(sid):
    return "\n".join("\n".join(lines) for p, lines in CLEANED[sid] if PAGE_STATUS[(sid, p)] == "content")

CONFIGS = {
    "fixed-500 (lab)":              lambda sid: fixed_chunks(whole_text(sid), 500),
    "lab paragraph-aware-500":      lambda sid: lab_paragraph_chunks(whole_text(sid), 500),
    "fixed-500, overlap 100":       lambda sid: fixed_chunks(whole_text(sid), 500, 100),
    "structure-aware-500":          lambda sid: [c["text"] for c in chunk_source(sid, 500)],
    "structure-aware-800":          lambda sid: [c["text"] for c in chunk_source(sid, 800)],
    "structure-aware-800, overlap 1 para": lambda sid: [c["text"] for c in chunk_source(sid, 800, 1)],
    "structure-aware-1000":         lambda sid: [c["text"] for c in chunk_source(sid, 1000)],
}

source_chars = sum(len(whole_text(s)) for s in DOC_SOURCES)
COMPARISON = []
print(f"{'configuration':<38}{'chunks':>7}{'mean':>6}{'max':>6}{'<150':>6}{'mid-start':>10}{'rules kept':>11}{'extra':>7}")
for name, fn in CONFIGS.items():
    per = {sid: fn(sid) for sid in DOC_SOURCES}
    chunks = [c for cs in per.values() for c in cs]
    lens = [len(c) for c in chunks]
    mid = sum(1 for c in chunks if c[0].islower() or (not c[0].isalnum() and c[0] not in "•-(“\""))
    kept = sum(1 for sid, parts in KEY_FACTS if any(all(p in norm(c) for p in parts) for c in per[sid]))
    dup = sum(lens) / source_chars - 1
    COMPARISON.append(dict(config=name, chunks=len(chunks), mean=round(st.mean(lens)), max=max(lens),
                           tiny=sum(l < 150 for l in lens), mid_start=mid, rules_kept=kept, extra_text=round(dup, 3)))
    print(f"{name:<38}{len(chunks):>7}{st.mean(lens):>6.0f}{max(lens):>6}{sum(l < 150 for l in lens):>6}{mid:>10}{kept:>7}/{len(KEY_FACTS)}{dup:>+7.0%}")

identical = all(fixed_chunks(whole_text(s), 500) == lab_paragraph_chunks(whole_text(s), 500) for s in DOC_SOURCES)
print("\nLab paragraph-aware splitter produces exactly the fixed-size chunks:", identical)
print("Blank-line paragraph breaks found in the extracted text:", sum(len(re.findall(r"\n\s*\n", whole_text(s))) for s in DOC_SOURCES))

configuration                          chunks  mean   max  <150 mid-start rules kept  extra
fixed-500 (lab)                            45   469   500     3        35      6/10    +0%
lab paragraph-aware-500                    45   469   500     3        35      6/10    +0%
fixed-500, overlap 100                     54   481   500     2        43      8/10   +23%
structure-aware-500                        60   388   500     4         0      9/10   +10%
structure-aware-800                        43   516   798     1         0      9/10    +5%
structure-aware-800, overlap 1 para        43   547   798     0         0     10/10   +11%
structure-aware-1000                       33   652   999     0         0     10/10    +2%

Lab paragraph-aware splitter produces exactly the fixed-size chunks: True
Blank-line paragraph breaks found in the extracted text: 0


In [10]:
# B3 (Pathway A): what the difference looks like on one rule (S05, the 800 kWh exemption)
fixed = fixed_chunks(whole_text("S05"), 500)
hit = next(c for c in fixed if "800kWj dan ke" in norm(c))
print("FIXED-500 chunk holding the rule:\n", hit, "\n")
chosen = next(c for c in chunk_source("S05", 1000) if "800kWj dan ke" in norm(c["text"]))
print(f"STRUCTURE-AWARE-1000 chunk (page {chosen['page']}, section: {chosen['section'][:60]}...):\n", chosen["text"])

FIXED-500 chunk holding the rule:
 LEBIH LAPAN JUTA PENGGUNA DOMESTIK TERIMA MANFAAT
MELALUI PENINGKATAN ARAS PERLINDUNGAN BIL ELEKTRIK
Tenaga Nasional Berhad (TNB) menyambut baik keputusan kerajaan dalam terus
memperkukuhkan mekanisme perlindungan buat pengguna domestik elektrik dengan
menaikkan aras perlindungan daripada 600kWj sebulan kepada 800kWj sebulan.
Melalui keputusan ini, kesemua pengguna domestik TNB dengan penggunaan 800kWj dan ke
bawah sebulan akan dikecualikan daripada dikenakan caj Pelarasan Kos Bahan Api Secara
A 

STRUCTURE-AWARE-1000 chunk (page 1, section: LEBIH LAPAN JUTA PENGGUNA DOMESTIK TERIMA MANFAAT MELALUI PE...):
 LEBIH LAPAN JUTA PENGGUNA DOMESTIK TERIMA MANFAAT MELALUI PENINGKATAN ARAS PERLINDUNGAN BIL ELEKTRIK
Tenaga Nasional Berhad (TNB) menyambut baik keputusan kerajaan dalam terus memperkukuhkan mekanisme perlindungan buat pengguna domestik elektrik dengan menaikkan aras perlindungan daripada 600kWj sebulan kepada 800kWj sebulan.
Melalui keputusan ini,

**Decision: structure-aware chunking, 1,000 characters, no overlap, one page at a time.**

- **Observation.** pypdf output has no blank lines between paragraphs, so the lab's paragraph-aware splitter finds no paragraphs and silently produces the same chunks as fixed-size. Fixed-size chunks start mid-word or mid-sentence most of the time and keep only 6 of the 10 rules whole. Adding a 100-character overlap brings that to 8 of 10, at the cost of duplicated text.
- **Selected.** Rebuild headings and paragraphs from line structure, pack whole paragraphs under their section heading up to 1,000 characters, never cross a heading or a page, no overlap.
- **Why 1,000 and not 500 or 800.** At 500 and 800, the S05 exemption is separated from its effective period because they sit in adjacent paragraphs under a long title. A one-paragraph overlap at 800 also keeps all 10, but adds 11% extra text. 1,000 keeps all 10 with no overlap and only 2% extra text (the repeated section headings). A chunk never spans two sections, so the larger size does not mix topics.
- **Accepted limitation.** Bigger chunks send more text to the model per retrieved passage (NFR4 cost) and may lower retrieval precision. This will be measured against retrieval results in Topic 7. Splitting at page boundaries is required for page citations (FR5), but a paragraph that runs across a page break is split in two.

In [11]:
# Cell 5A2: export traceable chunk records
CHUNK_SIZE, OVERLAP_PARAS = 1000, 0
PROCESSING_A = (f"pypdf {__import__('pypdf').__version__} layout extraction; cleaned: whitespace, page furniture, "
                f"ST letterhead and footer, text after end marker; structure-aware chunks; max_chars={CHUNK_SIZE}; "
                f"overlap_paragraphs={OVERLAP_PARAS}; page-bounded")

REVIEW_NOTES = {
    ("S05", 1): "Extraction artifact in source text: 'ko s bahan' should read 'kos bahan'",
    ("S06", 1): "Eligibility table flattened to text; each row keeps its order, columns read left to right",
}
SUPERSEDED = {
    "S03": "Its 600 kWh threshold is superseded for September to December 2026 usage by S05 and S06",
    "S04": "Its 600 kWh threshold is superseded for September to December 2026 usage by S05 and S06",
}

def build_records(sid):
    meta, digest = REGISTRY[sid], source_hash(source_path(sid))
    records = []
    for n, c in enumerate(chunk_source(sid, CHUNK_SIZE, OVERLAP_PARAS), 1):
        notes = [x for x in (REVIEW_NOTES.get((sid, c["page"])), SUPERSEDED.get(sid)) if x]
        records.append({
            "chunk_id": f"{sid}-p{c['page']}-c{n:02d}",
            "text": c["text"],
            "source_id": sid,
            "source_file": f"data/raw/{meta['filename']}",
            "source_url": meta["url"],
            "issuer": meta["issuer"],
            "title": meta["title"],
            "doc_ref": meta["doc_ref"],
            "language": meta["language"],
            "published_date": meta["published_date"],
            "effective_from": meta["effective_from"],
            "effective_to": meta["effective_to"] or None,
            "page": c["page"],
            "section": c["section"],
            "processing": PROCESSING_A,
            "source_sha256": digest,
            "status": "needs_review" if (sid, c["page"]) in REVIEW_NOTES else "ready",
            "notes": "; ".join(notes) or None,
        })
    return records

records = [r for sid in DOC_SOURCES for r in build_records(sid)]
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
with prepared_path.open("w", encoding="utf-8") as fh:
    for r in records:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Chunks exported:", len(records), "->", prepared_path.relative_to(BASE_DIR))
print("Per source:", {s: sum(r["source_id"] == s for r in records) for s in DOC_SOURCES})
print("Status:", {s: sum(r["status"] == s for r in records) for s in ("ready", "needs_review")})
print("\nSample record:")
print(json.dumps(records[0], ensure_ascii=False, indent=1)[:1400])

Chunks exported: 33 -> data/processed/prepared_text.jsonl
Per source: {'S02': 9, 'S03': 7, 'S04': 7, 'S05': 6, 'S06': 4}
Status: {'ready': 27, 'needs_review': 6}

Sample record:
{
 "chunk_id": "S02-p1-c01",
 "text": "Jadual Elektrik Baharu: Lebih 23.6 Juta Pengguna Domestik Semenanjung Nikmati Kadar Lebih Adil\nPutrajaya, 20 Jun 2025 – Lebih 23.6 juta pengguna domestik di Semenanjung akan menikmati kadar elektrik yang lebih adil dan progresif menerusi pelaksanaan jadual tarif elektrik baharu yang telah diluluskan oleh Kerajaan. Penetapan tarif ini akan berkuat kuasa mulai 1 Julai 2025 hingga 31 Disember 2027 di bawah kerangka kerja Kawal Selia Berasaskan Insentif (Incentive-Based Regulation - IBR), selaras dengan peruntukan Seksyen 26, Akta Bekalan Elektrik 1990.\nPerubahan tarif elektrik bagi Tempoh Regulatori Keempat (Regulatory Period 4 - RP4) melibatkan penstrukturan semula tiga (3) komponen berikut:\n(i) kadar purata tarif asas;\n(ii) jadual tarif baharu; dan\n(iii) mekanisme pela

In [12]:
# B3 (Pathway A): output contract checks, then read back what was written
exported = [json.loads(l) for l in prepared_path.read_text(encoding="utf-8").splitlines() if l.strip()]
REQUIRED = ["chunk_id", "text", "source_id", "source_file", "issuer", "title", "language",
            "published_date", "effective_from", "page", "processing", "source_sha256", "status"]
PHONE = re.compile(r"\b01\d-?\d{7,8}\b")

assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert len(exported) == len(records), "Exported count differs from prepared records"
assert len({r["chunk_id"] for r in exported}) == len(exported), "Duplicate chunk_id"
assert all(r["text"].strip() for r in exported), "Empty chunk text"
assert all(r[k] not in (None, "") for r in exported for k in REQUIRED), "Missing required metadata"
assert all(1 <= r["page"] <= int(REGISTRY[r["source_id"]]["pages"]) for r in exported), "Page outside source"
assert all(len(r["text"]) <= CHUNK_SIZE for r in exported), "Chunk over size limit"
assert {r["source_id"] for r in exported} == set(DOC_SOURCES), "A source produced no chunks"
assert not any(PHONE.search(r["text"]) for r in exported), "Personal phone number found in prepared text (RAI4)"
assert not any((r["source_id"], r["page"]) in PAGE_STATUS and PAGE_STATUS[(r["source_id"], r["page"])] != "content" for r in exported)
rules_whole = sum(1 for sid, parts in KEY_FACTS
                  if any(all(p in norm(r["text"]) for p in parts) for r in exported if r["source_id"] == sid))
assert rules_whole == len(KEY_FACTS), f"Only {rules_whole}/{len(KEY_FACTS)} key rules kept whole with their qualifiers"

print("Output:", prepared_path.relative_to(BASE_DIR), "Size:", prepared_path.stat().st_size, "bytes")
print(f"Key rules kept whole: {rules_whole}/{len(KEY_FACTS)}")
print("Validation: PASS. Contract checks passed; read a few records below as well.\n")
for r in exported[5:7]:
    print(f"[{r['chunk_id']}] {r['language']} p{r['page']} | {r['section']}")
    print(r["text"][:400], "\n")

Output: data/processed/prepared_text.jsonl Size: 53234 bytes
Key rules kept whole: 10/10
Validation: PASS. Contract checks passed; read a few records below as well.

[S02-p3-c06] ms p3 | Perubahan Struktur Tarif Elektrik Merangkumi Kategori Pengguna, Caj dan Insentif
Perubahan Struktur Tarif Elektrik Merangkumi Kategori Pengguna, Caj dan Insentif
Susulan pelaksanaan jadual tarif baharu ini, majoriti pengguna dijangka tidak terkesan.
Malah, pengguna yang mengamalkan penggunaan tenaga secara cekap akan menikmati penjimatan yang lebih tinggi menerusi Insentif Cekap Tenaga. 

[S02-p3-c07] ms p3 | Pembaharuan Mekanisme Pelarasan Kos
Pembaharuan Mekanisme Pelarasan Kos
Bagi komponen caj tenaga, mekanisme pelarasan kos bahan api baharu yang lebih dinamik iaitu Automatic Fuel Adjustment (AFA) akan menggantikan mekanisme Imbalance Cost Pass-Through (ICPT). Mekanisme ini membolehkan pelarasan kos penjanaan dilaksanakan secara automatik berdasarkan harga bahan api di pasaran semasa dan kadar tuka

### Pathway B: Tariff Rate Table

The rates are a table in S01. Two ways to get them into data:

- **Option A:** parse page 1 of S01 automatically with a pattern for `label unit value`.
- **Option B:** transcribe the rows in scope into a CSV by hand, with source page and printed label, then verify every value against the PDF text automatically.

Cell 5B runs both on the same page.

In [13]:
# Cell 5B (part 1): option A, automatic extraction of S01 page 1
import pandas as pd

s01_lines = page_texts(source_path("S01"), mode="plain")[0]
ROW = re.compile(r"^(Caj .+?) (sen/kWj|RM/bulan|RM/KW|RM/kW) ([\d.]+)$")
label_lines = [l for l in s01_lines if l.startswith("Caj ")]
parsed = [l for l in label_lines if ROW.match(l)]
lost = [l for l in label_lines if not ROW.match(l)]
print(f"Charge rows on page 1: {len(label_lines)}; parsed with a value: {len(parsed)}; value lost: {len(lost)}")
for l in lost:
    print("  no value on its line:", l)
section = next(i for i, l in enumerate(s01_lines) if l.startswith("Bukan Domestik Voltan Rendah Am"))
print("\nWhere the values went (reading order breaks inside this block):")
for l in s01_lines[section:section + 9]:
    print("  |", l)

Charge rows on page 1: 21; parsed with a value: 17; value lost: 4
  no value on its line: Caj Tenaga - Untuk semua kWj
  no value on its line: Caj Kapasiti - Untuk semua kWj
  no value on its line: Caj Rangkaian - Untuk semua kWj
  no value on its line: Caj Peruncitan

Where the values went (reading order breaks inside this block):
  | Bukan Domestik Voltan Rendah Am
  | Caj Tenaga - Untuk semua kWj
  | Caj Kapasiti - Untuk semua kWj
  | Caj Rangkaian - Untuk semua kWj
  | Caj Peruncitan
  | Bukan Domestik Voltan Rendah ToU
  | sen/kWj 27.03
  | sen/kWj 8.83
  | sen/kWj 14.82


In [14]:
# Cell 5B (part 2): option B, load the curated tables and inspect them
rates_src = select_source("tariff_domestic_rp4.csv")
rules_src = select_source("relief_rules.csv")
rates_raw = pd.read_csv(rates_src, dtype=str, keep_default_na=False)
rules_raw = pd.read_csv(rules_src, dtype=str, keep_default_na=False)

for name, df in (("tariff_domestic_rp4.csv", rates_raw), ("relief_rules.csv", rules_raw)):
    print(f"--- {name}: {df.shape[0]} rows x {df.shape[1]} columns")
    print("Empty cells per column:", {c: int((df[c] == "").sum()) for c in df.columns if (df[c] == "").any()} or "none")
    print("Exact duplicate rows:", int(df.duplicated().sum()))
    print(df.drop(columns=[c for c in ("label_in_source", "note", "description") if c in df]).to_string(index=False), "\n")

--- tariff_domestic_rp4.csv: 5 rows x 12 columns
Empty cells per column: none
Exact duplicate rows: 0
rate_id      tariff component         condition period  rate     unit effective_from effective_to source_id page
    R01 Domestik Am    energy usage <= 1500 kWh    all 27.03  sen/kWh     2025-07-01   2027-12-31       S01    1
    R02 Domestik Am    energy  usage > 1500 kWh    all 37.03  sen/kWh     2025-07-01   2027-12-31       S01    1
    R03 Domestik Am  capacity           all kWh    all  4.55  sen/kWh     2025-07-01   2027-12-31       S01    1
    R04 Domestik Am   network           all kWh    all 12.85  sen/kWh     2025-07-01   2027-12-31       S01    1
    R05 Domestik Am    retail         per month    all 10.00 RM/month     2025-07-01   2027-12-31       S01    1 

--- relief_rules.csv: 6 rows x 11 columns
Empty cells per column: {'effective_to': 2, 'note': 1}
Exact duplicate rows: 0
rule_id                    rule value      unit applies_to effective_from effective_to source_id 

In [15]:
# Cell 5B (part 3): verify every value against its source page, type the fields, export
EVIDENCE = {
    "P01": "consumption of 600kWh and below are fully exempted from AFA charges",
    "P02": "consumption of 800 kWh and below from September to 31 December 2026 will enjoy full exemption from:",
    "P03": "Pengguna dengan penggunaan 600 kWJ dan ke bawah dikecualikan Caj Peruncitan",
    "P04": "The AFA rate for September 2026 is +3.67 sen/kWh.",
    "P05": "exempted from the 8% service tax",
    "P06": "Caj Tenaga - Untuk penggunaan >1500 kWj",
}
UNITS = {"sen/kWh", "RM/month", "kWh/month", "percent"}

def page_text(sid, page, mode):
    return norm(" ".join(page_texts(source_path(sid), mode)[page - 1]))

def domestik_am_block():
    start = s01_lines.index("Domestik Am")
    end = next(i for i, l in enumerate(s01_lines) if l.startswith("Domestik ToU"))
    return s01_lines[start:end]

block = domestik_am_block()
rates = rates_raw.copy()
rates["verified_in_source"] = [any(l.startswith(r.label_in_source) and l.endswith(" " + r.rate) for l in block)
                               for r in rates.itertuples()]
rates["rate"] = rates["rate"].astype(float)
rates["page"] = rates["page"].astype(int)
rates["effective_from"] = pd.to_datetime(rates["effective_from"]).dt.date
rates["effective_to"] = pd.to_datetime(rates["effective_to"]).dt.date
rates["rate_rm"] = (rates["rate"] / 100).where(rates["unit"] == "sen/kWh", rates["rate"]).round(6)

rules = rules_raw.copy()
rules["evidence_text"] = rules["rule_id"].map(EVIDENCE)
rules["verified_in_source"] = [
    r.evidence_text in page_text(r.source_id, int(r.page), "plain" if r.source_id == "S01" else "layout")
    for r in rules.itertuples()]
rules["value"] = pd.to_numeric(rules["value"])
rules["page"] = rules["page"].astype(int)
rules["effective_from"] = pd.to_datetime(rules["effective_from"]).dt.date
rules["effective_to"] = pd.to_datetime(rules["effective_to"], errors="coerce").dt.date
rules["open_ended"] = rules["effective_to"].isna()

checks = {
    "rate keys unique": not rates.duplicated(["tariff", "component", "condition", "period"]).any(),
    "rule ids unique": rules["rule_id"].is_unique,
    "units allowed": set(rates["unit"]) | set(rules["unit"]) <= UNITS,
    "sources registered": set(rates["source_id"]) | set(rules["source_id"]) <= set(REGISTRY),
    "pages within source": all(p <= int(REGISTRY[s]["pages"]) for s, p in zip(rules["source_id"], rules["page"])),
    "dates ordered": all(pd.isna(t) or f <= t for f, t in zip(rules["effective_from"], rules["effective_to"])),
    "every rate found in S01 page 1": bool(rates["verified_in_source"].all()),
    "every rule found on its page": bool(rules["verified_in_source"].all()),
}
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
assert all(checks.values()), "Pathway B validation failed"

overlaps = rules[rules["rule"] == "exemption threshold"][["rule_id", "value", "effective_from", "effective_to"]]
print("\nExemption thresholds (P02 overrides P01 inside its period; nothing in the sources states what applies after it):")
print(overlaps.to_string(index=False))

prepared_rates = PROCESSED_DIR / "prepared_tariff_rates.csv"
prepared_rules = PROCESSED_DIR / "prepared_relief_rules.csv"
rates.to_csv(prepared_rates, index=False)
rules.to_csv(prepared_rules, index=False)
print("\nSaved:", prepared_rates.relative_to(BASE_DIR), "and", prepared_rules.relative_to(BASE_DIR))

  PASS  rate keys unique
  PASS  rule ids unique
  PASS  units allowed
  PASS  sources registered
  PASS  pages within source
  PASS  dates ordered
  PASS  every rate found in S01 page 1
  PASS  every rule found on its page

Exemption thresholds (P02 overrides P01 inside its period; nothing in the sources states what applies after it):
rule_id  value effective_from effective_to
    P01  600.0     2025-07-01          NaT
    P02  800.0     2026-09-01   2026-12-31

Saved: data/processed/prepared_tariff_rates.csv and data/processed/prepared_relief_rules.csv


In [16]:
# B3 (Pathway B): output contract, read back and compare with the raw CSVs
for raw_df, path in ((rates_raw, prepared_rates), (rules_raw, prepared_rules)):
    back = pd.read_csv(path)
    assert path.stat().st_size > 0 and len(back) == len(raw_df), f"{path.name}: row count changed"
    assert back["verified_in_source"].all(), f"{path.name}: unverified rows"
    added = [c for c in back.columns if c not in raw_df.columns]
    print(f"{path.name}: {len(back)} rows, same as input; added columns {added}")
    print("  dtypes:", {c: str(t) for c, t in back.dtypes.items() if c in ("rate", "rate_rm", "value", "page")})
print("\nValidation: PASS")
print("\nB3 comparison (Pathway B):")
print(f"  Option A, automatic: {len(parsed)}/{len(label_lines)} page 1 rows parsed; {len(lost)} rows lost their values with no error raised")
print(f"  Option B, curated + verified: {len(rates)}/{len(rates)} rates and {len(rules)}/{len(rules)} rules found word for word on their cited page")

prepared_tariff_rates.csv: 5 rows, same as input; added columns ['verified_in_source', 'rate_rm']
  dtypes: {'rate': 'float64', 'page': 'int64', 'rate_rm': 'float64'}
prepared_relief_rules.csv: 6 rows, same as input; added columns ['evidence_text', 'verified_in_source', 'open_ended']
  dtypes: {'value': 'float64', 'page': 'int64'}

Validation: PASS

B3 comparison (Pathway B):
  Option A, automatic: 17/21 page 1 rows parsed; 4 rows lost their values with no error raised
  Option B, curated + verified: 5/5 rates and 6/6 rules found word for word on their cited page


**Decision (Pathway B):** curated CSV, verified automatically against the source text.

- Automatic extraction parsed the Domestik Am rows correctly, but on the same page it lost the values of a whole tariff block without raising any error: a silent failure. A rate table that is wrong without anyone knowing breaks NFR1.
- The curated table is small (5 rates, 6 rules), so transcribing it is cheap. The verification step turns the main risk of ADR1, a transcription error, into a check that fails loudly.
- **Accepted limitation.** A human still types each new row, and verification proves the value is printed on the cited page, not that it was put in the right column. Independent reference bills (REF-CALC) remain the final check in Topic 8.

In [17]:
# B4.1: register lineage for every source used
for sid in DOC_SOURCES:
    register_lineage(source=source_path(sid), output=prepared_path, processing=PROCESSING_A)

register_lineage(rates_src, prepared_rates,
    "Curated transcription of S01 page 1 (Domestik Am); typed fields; each rate verified against S01 page 1 text")
register_lineage(rules_src, prepared_rules,
    "Curated rules from S01, S03 and S06; typed fields and dates; each rule verified against the evidence text on its cited page")

Registered: st_kenyataan_media_2025-06-20.pdf -> prepared_text.jsonl
Registered: tnb_20260915_23_bi.pdf -> prepared_text.jsonl
Registered: tnb_20260915_23_bm.pdf -> prepared_text.jsonl
Registered: tnb_20260918_24_bm.pdf -> prepared_text.jsonl
Registered: tnb_20261002_25_bi.pdf -> prepared_text.jsonl
Registered: tariff_domestic_rp4.csv -> prepared_tariff_rates.csv
Registered: relief_rules.csv -> prepared_relief_rules.csv


In [18]:
# Cell 6A: common lineage manifest, with version and rights taken from the registry
assert lineage_entries, "Register at least one pathway first."
by_file = {r["filename"]: r for r in REGISTRY.values()}
CURATED_FROM = {"tariff_domestic_rp4.csv": "S01", "relief_rules.csv": "S01, S03, S06"}

lineage_records = []
for i, e in enumerate(lineage_entries.values(), 1):
    src, out = e["source"], e["output"]
    assert src.is_file() and out.is_file() and out.stat().st_size > 0
    reg = by_file.get(src.name)
    lineage_records.append({
        "lineage_id": f"L{i:02d}",
        "source_id": reg["source_id"] if reg else f"curated ({CURATED_FROM[src.name]})",
        "source_filename": str(src.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(src),
        "prepared_output": str(out.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(out),
        "processing_note": e["processing"],
        "source_version": f"published {reg['published_date']}, retrieved {reg['retrieved_date']}" if reg else "v1, 2026-11-16",
        "access_rights": "Public official publication; raw file not redistributed in the repo" if reg else "Own work",
        "status": "Prepared; quality review in Topic 4",
    })

manifest = save_manifest(lineage_records)
print("Total lineage entries:", len(manifest["records"]))

Saved lineage: data/week3_manifest.json
Total lineage entries: 7


In [19]:
# Cell 6B: verify the saved manifest
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
saved = json.loads(manifest_path.read_text(encoding="utf-8"))["records"]
assert saved and len(saved) == len(lineage_entries)
for rec in saved:
    src, out = RAW_DIR / rec["source_filename"], BASE_DIR / rec["prepared_output"]
    assert src.is_file() and out.is_file()
    assert source_hash(src) == rec["source_sha256"], f"Source checksum mismatch: {src}"
    assert source_hash(out) == rec["prepared_sha256"], f"Output checksum mismatch: {out}"
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved))
print("Manifest:", manifest_path.relative_to(BASE_DIR))

COMMON LINEAGE VALIDATION: PASS
Verified records: 7
Manifest: data/week3_manifest.json


## Part C: Build and Document the V1 Data Pipeline

Cell 7 is the lab's common pipeline runner, changed in two ways: it takes **several input files** (Pathway A has five PDFs) and writes paths **relative to the project folder**, so manifests stay valid after the files are copied into the repository. Cells 8A and 8B reuse the Part B functions instead of copying them, so Parts B and C cannot drift apart.

In [20]:
# Cell 7: common V1 pipeline utilities
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_RUNS = {}
rel = lambda p: str(Path(p).resolve().relative_to(BASE_DIR.resolve()))

def execute_pipeline(kind, sources, inspect_fn, prepare_fn, configuration):
    inputs = [Path(p) for p in sources]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")
    inspection = inspect_fn(inputs, configuration)
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]
    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for out in outputs:
        if not out.is_file() or out.stat().st_size == 0:
            raise ValueError(f"Prepared output missing or empty: {out}")
    validation = {"output_count": len(outputs),
                  "output_sha256": [sha256(p) for p in outputs],
                  "output_bytes": [p.stat().st_size for p in outputs]}
    record = {"pipeline_version": "week3-v1",
              "generated_at_utc": datetime.now(timezone.utc).isoformat(),
              "data_type": kind, "configuration": configuration, "processing_method": method,
              "sources": [{"path": rel(p), "sha256": sha256(p)} for p in inputs],
              "outputs": [rel(p) for p in outputs],
              "source_output_mapping": mappings, "inspection": inspection, "validation": validation,
              "source_version": "See data/raw/source_registry.csv",
              "access_rights": "Public official publications; raw PDFs not redistributed"}
    manifest = output_dir / "pipeline_manifest.json"
    manifest.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", rel(manifest))
    return record

def register_and_run(kind, sources, inspect_fn, prepare_fn, configuration):
    runner = lambda: execute_pipeline(kind, sources, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Cell 7 ready. Run only the selected Cell 8 pathways.")

Cell 7 ready. Run only the selected Cell 8 pathways.


In [21]:
# Cell 8A: documents pathway (same method and configuration as Part B)
SOURCES_A = [source_path(sid) for sid in DOC_SOURCES]
CONFIG_A = {"method": "structure-aware", "chunk_size": CHUNK_SIZE, "overlap_paragraphs": OVERLAP_PARAS,
            "extraction": "pypdf layout", "page_bounded": True}

def inspect_documents(inputs, cfg):
    report = {}
    for p in inputs:
        sid = by_file[p.name]["source_id"]
        cleaned, log = clean_pages(page_texts(p))
        if not any(sum(len(l) for l in lines) >= MIN_PAGE_CHARS for _, lines in cleaned):
            raise ValueError(f"{p.name}: no extractable text. Scanned PDFs need OCR.")
        report[sid] = {"pages": len(cleaned), "lines_removed": log,
                       "pages_without_text": [n for n, lines in cleaned if sum(len(l) for l in lines) < MIN_PAGE_CHARS]}
    return report

def prepare_documents(inputs, out, cfg):
    target = out / "prepared_text.jsonl"
    rows = []
    for p in inputs:
        sid = by_file[p.name]["source_id"]
        CLEANED[sid], _ = clean_pages(page_texts(p))
        rows += build_records(sid)
    with target.open("w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    mapping = [{"source": rel(p), "output": rel(target),
                "chunks": sum(r["source_id"] == by_file[p.name]["source_id"] for r in rows)} for p in inputs]
    return [target], mapping, f"structure-aware chunks of up to {cfg['chunk_size']} characters, page-bounded, {len(rows)} chunks"

run_A = register_and_run("documents", SOURCES_A, inspect_documents, prepare_documents, CONFIG_A)
print("Matches Part B output:", sha256(PIPE_DIR / "documents" / "prepared_text.jsonl") == sha256(prepared_path))

SUCCESS: documents | Inputs: 5 | Outputs: 1
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
Matches Part B output: True


In [22]:
# Cell 8B: tariff table pathway (same verification as Part B)
SOURCES_B = [rates_src, rules_src]
CONFIG_B = {"method": "curated CSV verified against source PDF text", "scope": "Domestik Am", "verify": True}

def inspect_tables(inputs, cfg):
    report = {}
    for p in inputs:
        df = pd.read_csv(p, dtype=str, keep_default_na=False)
        if df.empty:
            raise ValueError(f"{p.name} is empty")
        report[p.name] = {"rows": len(df), "columns": list(df.columns), "duplicates": int(df.duplicated().sum())}
    return report

def prepare_tables(inputs, out, cfg):
    targets = []
    for p, frame in zip(inputs, (rates, rules)):
        if not frame["verified_in_source"].all():
            raise ValueError(f"{p.name}: rows not verified against source")
        target = out / ("prepared_" + p.name.replace("_domestic_rp4", "_rates"))
        frame.to_csv(target, index=False)
        targets.append(target)
    mapping = [{"source": rel(s), "output": rel(t)} for s, t in zip(inputs, targets)]
    return targets, mapping, "typed and verified: 5 Domestik Am rates from S01 page 1, 6 rules from S01, S03, S06"

run_B = register_and_run("tables", SOURCES_B, inspect_tables, prepare_tables, CONFIG_B)
print("Matches Part B output:", [sha256(PIPE_DIR / "tables" / n) == sha256(PROCESSED_DIR / n)
                                  for n in ("prepared_tariff_rates.csv", "prepared_relief_rules.csv")])

SUCCESS: tables | Inputs: 2 | Outputs: 2
Manifest: data/pipeline_v1/tables/pipeline_manifest.json
Matches Part B output: [True, True]


In [23]:
# Cell 9: repeatability and provenance check (lab code)
assert PIPELINE_RUNS, "Run at least one Cell 8 pathway first."
print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first, second = entry["first"], entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Mapping changed for {kind}"
    print(f"PASS: {kind}: source fingerprints, prepared outputs and mappings match")
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---


SUCCESS: documents | Inputs: 5 | Outputs: 1
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents: source fingerprints, prepared outputs and mappings match
SUCCESS: tables | Inputs: 2 | Outputs: 2
Manifest: data/pipeline_v1/tables/pipeline_manifest.json
PASS: tables: source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


In [24]:
# Optional (Colab): zip the outputs for download into the repository
if "google.colab" in sys.modules:
    import shutil
    archive = shutil.make_archive("/content/week3_outputs", "zip", BASE_DIR / "data",
                                  base_dir=".")
    from google.colab import files as colab_files
    colab_files.download(archive)
else:
    print("Local run: outputs are already in", BASE_DIR / "data")

Local run: outputs are already in /content/week3_project/data


### Outputs

| Artefact | Path |
|---|---|
| Prepared chunks (Part B) | `data/processed/prepared_text.jsonl` |
| Prepared rate table and rules (Part B) | `data/processed/prepared_tariff_rates.csv`, `prepared_relief_rules.csv` |
| Lineage manifest (Part B) | `data/week3_manifest.json` |
| Pipeline runs (Part C) | `data/pipeline_v1/documents/`, `data/pipeline_v1/tables/`, each with `pipeline_manifest.json` |

Decisions, stage contracts and limitations: `docs/week3_pipeline_decisions.md`. Sources and inspection evidence: `docs/week3_data_inventory.md`.